# SWIR Processed-Box Analysis: Train + Validation Center 10x10

Adapted from `SWIR_processed_box_analysis_center_bruised_10x10.ipynb` for this project. It uses only `data/splits/train` and `data/splits/val`; the test split is not read. All spectra are computed from the centered 10x10 patch of each processed box and plotted against mapped SWIR wavelength.

In [ ]:
from pathlib import Path
import gc, math, re

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from scipy import stats

PATCH = 10
TIMEPOINTS = ("t0", "t1", "t3")
TOP_K = 10
CODE_RE = re.compile(r"^MICROTEC_(\d+)_processed_boxes\.h5$")
REQUESTED_PREFIXES = ["28", "29", "280", "281", "282", "283", "284", "285", "290", "291", "292", "293", "294", "295"]

pd.set_option("display.max_colwidth", 160)
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

In [ ]:
def find_project_root(start=Path.cwd().resolve()):
    for path in [start, *start.parents]:
        if (path / "data" / "splits" / "train").exists() and (path / "data" / "splits" / "val").exists():
            return path
    raise FileNotFoundError("Could not find data/splits/train and data/splits/val.")


def find_wavelengths(root):
    candidates = [
        root / "references" / "swir_hippa_wl_calib_rounded.npy",
        root / "references" / "swir_hippa_wl_calib.npy",
        root.parents[1] / "bruising" / "references" / "swir_hippa_wl_calib_rounded.npy",
        root.parents[1] / "bruising" / "references" / "swir_hippa_wl_calib.npy",
    ]
    for path in candidates:
        if path.exists():
            return np.load(path).astype(float)
    raise FileNotFoundError("Could not find SWIR wavelength mapping.")


ROOT = find_project_root()
SPLIT_DIRS = [ROOT / "data" / "splits" / "train", ROOT / "data" / "splits" / "val"]
WAVELENGTHS_NM = find_wavelengths(ROOT)
REPORT_DIR = ROOT / "reports" / "tables"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(ROOT)

## Inventory and groups

In [ ]:
def code_from_path(path):
    m = CODE_RE.match(path.name)
    return m.group(1) if m else None


def roi_type(name):
    name = name.lower()
    if name.startswith("sound"):
        return "sound"
    if name.startswith("bruised"):
        return "bruised"
    return "unknown"

files = sorted(path for split_dir in SPLIT_DIRS for path in split_dir.glob("*.h5"))
rows = []
for path in files:
    code = code_from_path(path)
    with h5py.File(path, "r") as f:
        for tp in TIMEPOINTS:
            group_name = f"SWIR_{tp}"
            sound = bruised = 0
            bands = None
            if group_name in f:
                for ds_name, ds in f[group_name].items():
                    if not isinstance(ds, h5py.Dataset):
                        continue
                    sound += roi_type(ds_name) == "sound"
                    bruised += roi_type(ds_name) == "bruised"
                    bands = ds.shape[-1]
            rows.append({"sample_code": code, "file": path.name, "timepoint": tp, "sound_boxes": sound, "bruised_boxes": bruised, "bands": bands})

inventory = pd.DataFrame(rows)
complete_codes = sorted(code for code, g in inventory.groupby("sample_code") if (g["sound_boxes"].gt(0).all() and g["bruised_boxes"].gt(0).all()))
groups = {f"group_{p}": [code for code in complete_codes if code.startswith(p)] for p in REQUESTED_PREFIXES}
display(inventory.groupby("sample_code").agg(timepoints=("timepoint", "size"), min_sound=("sound_boxes", "min"), min_bruised=("bruised_boxes", "min"), bands=("bands", "max")).reset_index())
display(pd.DataFrame({"group": groups.keys(), "n_samples": [len(v) for v in groups.values()]}))

## Core helpers

In [ ]:
path_by_code = {code_from_path(p): p for p in files}


def sorted_roi_names(group, kind):
    names = [n for n, ds in group.items() if isinstance(ds, h5py.Dataset) and roi_type(n) == kind]
    return sorted(names, key=lambda n: (int(re.search(r"(\d+)$", n).group(1)) if re.search(r"(\d+)$", n) else 0, n))


def center10(cube):
    h, w, _ = cube.shape
    y0, x0 = h // 2 - PATCH // 2, w // 2 - PATCH // 2
    return cube[y0:y0 + PATCH, x0:x0 + PATCH, :]


def load_spectra(code, tp, kind):
    with h5py.File(path_by_code[code], "r") as f:
        group = f[f"SWIR_{tp}"]
        spectra = []
        for name in sorted_roi_names(group, kind):
            spectra.append(center10(group[name][...]).reshape(-1, group[name].shape[-1]).mean(axis=0))
        return np.vstack(spectra)


def group_data(codes):
    sound_stack, bruised_stack = {}, {}
    for tp in TIMEPOINTS:
        sound_stack[tp] = np.vstack([load_spectra(code, tp, "sound") for code in tqdm(codes, desc=f"sound {tp}")])
        bruised_stack[tp] = np.vstack([load_spectra(code, tp, "bruised") for code in tqdm(codes, desc=f"bruised {tp}")])
    n_bands = next(iter(sound_stack.values())).shape[1]
    wl = WAVELENGTHS_NM[:n_bands]
    return {
        "codes": codes,
        "wavelengths": wl,
        "sound_stack": sound_stack,
        "bruised_stack": bruised_stack,
        "sound_mean": {tp: sound_stack[tp].mean(axis=0) for tp in TIMEPOINTS},
        "bruised_mean": {tp: bruised_stack[tp].mean(axis=0) for tp in TIMEPOINTS},
    }

## Two examples: central 10x10 red boxes

In [ ]:
example_code = next(code for code in complete_codes if code in path_by_code)
example_tp = "t3"
example_wl = 1292.0
example_idx = int(np.abs(WAVELENGTHS_NM - example_wl).argmin())

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
with h5py.File(path_by_code[example_code], "r") as f:
    group = f[f"SWIR_{example_tp}"]
    for ax, kind in zip(axes, ["sound", "bruised"]):
        name = sorted_roi_names(group, kind)[0]
        cube = group[name][...]
        h, w, _ = cube.shape
        y0, x0 = h // 2 - PATCH // 2, w // 2 - PATCH // 2
        ax.imshow(cube[:, :, example_idx], cmap="gray")
        ax.add_patch(Rectangle((x0, y0), PATCH, PATCH, fill=False, edgecolor="red", linewidth=2))
        ax.set_title(f"{kind}: {example_code} {example_tp}/{name}\n{WAVELENGTHS_NM[example_idx]:.1f} nm", fontsize=9)
        ax.axis("off")
plt.tight_layout()

## Plotting and ranking helpers

In [ ]:
def top_distinctive(g, top_k=TOP_K):
    rows = []
    wl = g["wavelengths"]
    for tp in [t for t in TIMEPOINTS if t != "t0"]:
        score = np.abs((g["bruised_mean"][tp] - g["sound_mean"][tp]) - (g["bruised_mean"]["t0"] - g["sound_mean"]["t0"]))
        for idx in np.argsort(score)[::-1][:top_k]:
            rows.append({"timepoint": tp, "wavelength_nm": wl[idx], "score": score[idx]})
    return pd.DataFrame(rows)


def top_nonoverlap(g, top_k=TOP_K):
    rows = []
    wl = g["wavelengths"]
    for tp in [t for t in TIMEPOINTS if t != "t0"]:
        s, b = g["sound_stack"][tp], g["bruised_stack"][tp]
        sm, bm, ss, bs = s.mean(0), b.mean(0), s.std(0), b.std(0)
        mask = (sm + ss < bm - bs) | (bm + bs < sm - ss)
        score = np.where(mask, np.abs(bm - sm), -np.inf)
        for idx in np.where(np.isfinite(score))[0][np.argsort(score[np.isfinite(score)])[::-1]][:top_k]:
            rows.append({"timepoint": tp, "wavelength_nm": wl[idx], "score": score[idx]})
    return pd.DataFrame(rows)


def plot_all(group_name, g):
    wl = g["wavelengths"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for ax, tp in zip(axes.ravel(), TIMEPOINTS):
        ax.plot(wl, g["sound_mean"][tp], label="sound", color="limegreen")
        ax.plot(wl, g["bruised_mean"][tp], label="bruised", color="crimson", ls="--")
        ax.fill_between(wl, g["sound_mean"][tp], g["bruised_mean"][tp], alpha=0.15)
        ax.set_title(f"{group_name} {tp}"); ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Mean reflectance")
    axes[0, 0].legend(); plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    for tp in [t for t in TIMEPOINTS if t != "t0"]:
        axes[0].plot(wl, g["bruised_mean"][tp] - g["bruised_mean"]["t0"], label=f"{tp}-t0")
        axes[1].plot(wl, (g["bruised_mean"][tp] - g["sound_mean"][tp]) - (g["bruised_mean"]["t0"] - g["sound_mean"]["t0"]), label=tp)
    axes[0].axhline(0, color="k", lw=1); axes[0].set_title("Bruised change from t0")
    axes[1].axhline(0, color="k", lw=1); axes[1].set_title("Distinctiveness vs t0")
    t = np.arange(len(TIMEPOINTS))
    axes[2].plot(t, [g["sound_mean"][tp].mean() for tp in TIMEPOINTS], "o-", label="sound")
    axes[2].plot(t, [g["bruised_mean"][tp].mean() for tp in TIMEPOINTS], "o--", label="bruised")
    axes[2].set_xticks(t, TIMEPOINTS); axes[2].set_title("Mean spectral intensity over time")
    for ax in axes: ax.set_xlabel("Wavelength (nm)" if ax is not axes[2] else "Timepoint"); ax.legend(); ax.grid(alpha=0.25)
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for ax, tp in zip(axes.ravel(), TIMEPOINTS):
        for kind, color in [("sound", "limegreen"), ("bruised", "crimson")]:
            stack = g[f"{kind}_stack"][tp]
            m, sd = stack.mean(0), stack.std(0)
            ax.plot(wl, m, color=color, label=kind)
            ax.fill_between(wl, m - sd, m + sd, color=color, alpha=0.15)
        ax.set_title(f"{group_name} variance {tp}"); ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Mean reflectance")
    axes[0, 0].legend(); plt.show()

## Primary groups: `28*` and `29*`

In [ ]:
PRIMARY_RESULTS = {}
for name in ["group_28", "group_29"]:
    if not groups[name]:
        continue
    print("= " * 35, f"\n{name}: {groups[name]}")
    g = group_data(groups[name])
    PRIMARY_RESULTS[name] = g
    plot_all(name, g)
    display(Markdown(f"**Top distinctive wavelengths: {name}**")); display(top_distinctive(g))
    display(Markdown(f"**Top non-overlap wavelengths: {name}**")); display(top_nonoverlap(g))

## Subgroups: `280*`-`285*` and `290*`-`295*`

In [ ]:
SUBGROUP_RESULTS = {}
for name in [f"group_{p}" for p in ["280", "281", "282", "283", "284", "285", "290", "291", "292", "293", "294", "295"]]:
    if not groups[name]:
        continue
    print("= " * 35, f"\n{name}: n={len(groups[name])}")
    g = group_data(groups[name])
    SUBGROUP_RESULTS[name] = g
    plot_all(name, g)
    display(Markdown(f"**Top distinctive wavelengths: {name}**")); display(top_distinctive(g))
    display(Markdown(f"**Top non-overlap wavelengths: {name}**")); display(top_nonoverlap(g))
    gc.collect()

## t0 paired healthy-vs-bruised comparison

In [ ]:
t0_rows = []
for code in complete_codes:
    s = load_spectra(code, "t0", "sound").mean(axis=0)
    b = load_spectra(code, "t0", "bruised").mean(axis=0)
    t0_rows.append({"sample_code": code, "sound_mean_t0": s.mean(), "bruised_mean_t0": b.mean(), "delta_bruised_minus_sound": (b - s).mean()})

t0_compare = pd.DataFrame(t0_rows).sort_values("sample_code")
paired_t = stats.ttest_rel(t0_compare["bruised_mean_t0"], t0_compare["sound_mean_t0"])
wilcoxon = stats.wilcoxon(t0_compare["bruised_mean_t0"], t0_compare["sound_mean_t0"], alternative="two-sided")
t0_tests = pd.DataFrame([
    {"test": "paired t-test", "n_samples": len(t0_compare), "statistic": paired_t.statistic, "p_value": paired_t.pvalue},
    {"test": "Wilcoxon signed-rank", "n_samples": len(t0_compare), "statistic": wilcoxon.statistic, "p_value": wilcoxon.pvalue},
])
display(t0_compare); display(t0_tests)

## Boxplots for one subgroup's strongest non-overlap wavelengths

In [ ]:
GROUP_FOR_BOXPLOTS = "group_280"
g = SUBGROUP_RESULTS.get(GROUP_FOR_BOXPLOTS) or PRIMARY_RESULTS.get(GROUP_FOR_BOXPLOTS)
if g is None:
    raise ValueError(f"Run group analysis first for {GROUP_FOR_BOXPLOTS}")
ranked = top_nonoverlap(g, top_k=6)
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for ax, (_, row) in zip(axes.ravel(), ranked.iterrows()):
    idx = int(np.abs(g["wavelengths"] - row["wavelength_nm"]).argmin())
    tp = row["timepoint"]
    ax.boxplot([g["sound_stack"][tp][:, idx], g["bruised_stack"][tp][:, idx]], labels=["sound", "bruised"])
    ax.set_title(f"{tp} | {row['wavelength_nm']:.1f} nm")
    ax.set_ylabel("Mean center-10x10 reflectance")
plt.show()

## Save compact summary tables

In [ ]:
summary_rows = []
for name, result in {**PRIMARY_RESULTS, **SUBGROUP_RESULTS}.items():
    for _, row in top_distinctive(result).iterrows():
        summary_rows.append({"group": name, "analysis": "distinctive_vs_t0", **row.to_dict()})
    for _, row in top_nonoverlap(result).iterrows():
        summary_rows.append({"group": name, "analysis": "non_overlap", **row.to_dict()})
summary = pd.DataFrame(summary_rows)
summary.to_csv(REPORT_DIR / "swir_train_val_center_10x10_group_wavelength_summary.csv", index=False)
t0_compare.to_csv(REPORT_DIR / "swir_train_val_center_10x10_t0_compare.csv", index=False)
t0_tests.to_csv(REPORT_DIR / "swir_train_val_center_10x10_t0_tests.csv", index=False)
summary.head()